# Seleção de candidatos a regex com base nos músculos

In [19]:
import re
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from fitlib.api import get_list

## 1. Obter os exercícios

In [20]:
raw_exercises = get_list("exerciseinfo/", {"language__code": "en", "limit": 100})

rows = []
for exercise in raw_exercises:
    translation = next((t for t in exercise.get("translations", []) if t.get("language") == 2), {})
    name = translation.get("name", "") or ""
    rows.append({
        "id": exercise["id"],
        "name": name,
        "category": (exercise.get("category") or {}).get("name", ""),
        "muscle_count": len(exercise.get("muscles") or []) + len(exercise.get("muscles_secondary") or []),
        "words": set(re.findall(r"[^\W\d_]+", name.lower()))
    })

exercises_df = pd.DataFrame(rows)
print("Total exercises:", len(exercises_df))
display(exercises_df[["id", "name", "category", "muscle_count"]].head(8))

Total exercises: 973


,id,name,category,muscle_count
0,9,2 Handed Kettlebell Swing,Abs,4
1,12,Seated Hip Adduction,Legs,1
2,20,Arnold Shoulder Press,Shoulders,3
3,31,Axe Hold,Arms,0
4,41,Barbell Ab Rollout,Abs,1
5,43,Barbell Hack Squats,Legs,3
6,46,Barbell Lunges Standing,Legs,2
7,48,Barbell Reverse Wrist Curl,Arms,0


## 2. Selecionar as palavras a analisar

In [21]:
exercises_df["words"] = exercises_df["name"].str.lower().apply(lambda name: re.findall(r"[a-z]+", name))

exploded_words = exercises_df.explode("words")

ignored_words = {
    # Articles, prepositions and connectors
    "a", "and", "at", "between", "for", "in",
    "into", "it", "no", "of", "off", "on",
    "or", "the", "through", "to", "using",
    "while", "with", "around", "behind", "away",

    # Quantities and generic modifiers
    "one", "two", "three", "four", "single",
    "double", "half", "full", "multi",

    # Less informative adjectives and adverbs
    "active", "alternative", "basic", "continuous",
    "controlled", "easy", "free", "gentle",
    "guided", "heavy", "modified", "negative",
    "slow", "standard", "slight", "small",
    "extreme", "loaded", "long", "brisk",

    # Generic positions and characteristics
    "standing", "seated", "sitting", "lying",
    "laying", "left", "right", "forward",
    "backward", "middle", "mid", "horizontal",
    "upright", "diagonal", "flat",

    # Generic and administrative words
    "custom", "exercise", "exercises",
    "workout", "workouts", "session", "sets",
    "training", "variation", "count", "width",
    "failure", "timed", "paused", "pause", "zone",

    # Connectors and modifiers in other languages
    "al", "alle", "con", "de", "mit",
    "alto", "alta", "ancho", "estrecho",
    "lento", "avanti", "seduto", "inclinado",

    # Abbreviations without clear classification meaning
    "db", "bb", "sz", "ez", "hd", "nb",
    "mp", "ol", "ss", "ir", "mm", "jm",
    "l", "m", "n", "s", "t", "v",
    "w", "y", "d", "i", "ytw", "ytwl",
    "ywts"

    # Word fragments and colors
    "eca", "mu", "jal",
    "yellow", "green", "black"
}

exploded_words = exploded_words[~exploded_words["words"].isin(ignored_words)]

word_frequencies = (exploded_words.groupby("words", as_index=False)
                    .agg(exercise_count=("id", "nunique"), example_name=("name", "first"))
                    .sort_values(["exercise_count", "words"], ascending=[False, True]))

display(word_frequencies)

,words,exercise_count,example_name
386,press,77,Arnold Shoulder Press
169,dumbbell,73,Bent Over Dumbbell Rows (Single Arm)
81,cable,72,Biceps Curl With Cable
520,stretch,62,Left neck stretch
302,leg,54,Single Leg Extension
...,...,...,...
591,windshield,1,Windshield Wipers
594,woodchoppers,1,Cable Woodchoppers
596,yoga,1,Yoga exercise: Cow-cat
597,ywts,1,YWTs


In [22]:
candidate_words = word_frequencies["words"].dropna().str.lower().unique()
print("Words to analyze:", len(candidate_words))

Words to analyze: 599


## 3. Calcular as proporções por palavra

In [23]:
valid_exercises = exercises_df[
    (exercises_df["category"].str.lower() != "cardio") &
    (exercises_df["muscle_count"] > 0)
]

evidence_rows = []
for term in candidate_words:
    matches = valid_exercises[valid_exercises["words"].apply(lambda words: term in words)]
    total = len(matches)
    if total == 0:
        continue

    one_muscle = int((matches["muscle_count"] == 1).sum())
    multiple_muscles = total - one_muscle

    evidence_rows.append({
        "term": term,
        "exercise_count": total,
        "one_muscle": one_muscle,
        "multiple_muscles": multiple_muscles,
        "one_muscle_pct": round(100 * one_muscle / total, 1),
        "multiple_muscles_pct": round(100 * multiple_muscles / total, 1),
        "example_name": matches.iloc[0]["name"],
        "id": matches.iloc[0]["id"]
    })

evidence_df = pd.DataFrame(evidence_rows).sort_values("exercise_count", ascending=False)
print("Exercises with valid data:", len(valid_exercises))
print("One muscle:", int((valid_exercises["muscle_count"] == 1).sum()))
print("Two or more muscles:", int((valid_exercises["muscle_count"] >= 2).sum()))
display(evidence_df.head(35))

Exercises with valid data: 789
One muscle: 277
Two or more muscles: 512


,term,exercise_count,one_muscle,multiple_muscles,one_muscle_pct,multiple_muscles_pct,example_name,id
0,press,75,23,52,30.7,69.3,Arnold Shoulder Press,20
1,dumbbell,68,23,45,33.8,66.2,Bent Over Dumbbell Rows (Single Arm),81
2,cable,67,44,23,65.7,34.3,Biceps Curl With Cable,95
4,leg,51,20,31,39.2,60.8,Single Leg Extension,71
6,pull,40,6,34,15.0,85.0,Close-grip Lat Pull Down,158
5,curl,39,21,18,53.8,46.2,Barbell Wrist Curl,51
8,squat,38,2,36,5.3,94.7,Braced Squat,124
7,row,37,11,26,29.7,70.3,Incline Dumbbell Row,310
3,stretch,36,15,21,41.7,58.3,Elevated prayer stretch,1027
10,raise,34,14,20,41.2,58.8,"Hip Raise, Lying",292


## 4. Aplicar critérios mínimos e sugerir candidatos

Definimos **pelo menos 5 exercícios por palavra** (para não decidir com base em apenas 1 ou 2 ocorrências) e **60% de concordância** com uma das duas faixas musculares. São parâmetros **exploratórios**, não limiares cientificamente validados; podem ser comparados com outros valores.

- `isolated`: 60% ou mais das ocorrências válidas com 1 músculo.
- `compound`: 60% ou mais das ocorrências válidas com 2+ músculos.
- **Não selecionado:** palavras raras ou sem predominância clara. **Não as colocar no `OTHER_PATTERN`**: falta de evidência não implica exercício do tipo *other*.

Os candidatos a `cardio` e a `other` **não podem ser derivados deste critério muscular**. Mantêm a lógica anterior: categoria Cardio e padrões próprios previamente analisados.

In [24]:
min_exercises = 6
min_agreement_pct = 66.67

evidence_df["suggestion"] = "not_selected"
evidence_df.loc[
    (evidence_df["exercise_count"] >= min_exercises) &
    (evidence_df["one_muscle_pct"] >= min_agreement_pct),
    "suggestion"
] = "isolated"
evidence_df.loc[
    (evidence_df["exercise_count"] >= min_exercises) &
    (evidence_df["multiple_muscles_pct"] >= min_agreement_pct),
    "suggestion"
] = "compound"

display(
    evidence_df[
        ["term", "exercise_count", "one_muscle_pct",
         "multiple_muscles_pct", "suggestion", "example_name"]
    ].head(40)
)

candidates = {
    "compound": evidence_df.loc[evidence_df["suggestion"] == "compound", "term"].tolist(),
    "isolated": evidence_df.loc[evidence_df["suggestion"] == "isolated", "term"].tolist()
}
print("Compound candidates:", candidates["compound"])
print("Isolated candidates:", candidates["isolated"])

,term,exercise_count,one_muscle_pct,multiple_muscles_pct,suggestion,example_name
0,press,75,30.7,69.3,compound,Arnold Shoulder Press
1,dumbbell,68,33.8,66.2,not_selected,Bent Over Dumbbell Rows (Single Arm)
2,cable,67,65.7,34.3,not_selected,Biceps Curl With Cable
4,leg,51,39.2,60.8,not_selected,Single Leg Extension
6,pull,40,15.0,85.0,compound,Close-grip Lat Pull Down
5,curl,39,53.8,46.2,not_selected,Barbell Wrist Curl
8,squat,38,5.3,94.7,compound,Braced Squat
7,row,37,29.7,70.3,compound,Incline Dumbbell Row
3,stretch,36,41.7,58.3,not_selected,Elevated prayer stretch
10,raise,34,41.2,58.8,not_selected,"Hip Raise, Lying"


Compound candidates: ['press', 'pull', 'squat', 'row', 'ups', 'up', 'push', 'bench', 'incline', 'front', 'trx', 'squats', 'plank', 'biceps', 'calf', 'kettlebell', 'deadlift', 'sit', 'down', 'hold', 'bent', 'wall', 'dips', 'split', 'wide', 'knee', 'lunges', 'floor', 'thrust', 'rings', 'clean', 'bridge', 'smith', 'rowing', 'narrow']
Isolated candidates: ['curls', 'extension', 'triceps', 'lateral', 'dumbbells', 'tricep', 'overhead', 'pushdown', 'preacher', 'kickback']


In [25]:
generated_patterns = {}

for effort_type, terms in candidates.items():
    if terms:
        escaped_terms = [re.escape(term) for term in terms]
        generated_patterns[effort_type] = r"\b(?:" + "|".join(escaped_terms) + r")\b"
    else:
        generated_patterns[effort_type] = None

    print(f"{effort_type.upper()}_PATTERN = {generated_patterns[effort_type]!r}")

COMPOUND_PATTERN = '\\b(?:press|pull|squat|row|ups|up|push|bench|incline|front|trx|squats|plank|biceps|calf|kettlebell|deadlift|sit|down|hold|bent|wall|dips|split|wide|knee|lunges|floor|thrust|rings|clean|bridge|smith|rowing|narrow)\\b'
ISOLATED_PATTERN = '\\b(?:curls|extension|triceps|lateral|dumbbells|tricep|overhead|pushdown|preacher|kickback)\\b'
